[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivamkalakaar/local-ai-lab-agents/blob/main/notebooks/04_retrieval_rag.ipynb)

# 04 · Retrieval (RAG): hybrid search with checked citations
**Local AI Lab** · CS329Z Lecture 3 · [Module 04](https://claude.ai/artifact/41sHaxGwFWf7SjJWDdtK5w#m4)

The default that works: split on structure, add a short context note to each chunk, search two ways (keywords and meaning), merge with reciprocal rank fusion, and check every citation in code. Everything runs locally, including the embedding model.

In [ ]:
# Local AI Lab setup: run this cell first (about 2-3 minutes on Colab, a few seconds locally)
import os, sys, subprocess
EXTRA = ['rank-bm25']
if "google.colab" in sys.modules:
    if not os.path.exists("/content/local-ai-lab-agents"):
        subprocess.run(["git", "clone", "-q", "https://github.com/shivamkalakaar/local-ai-lab-agents", "/content/local-ai-lab-agents"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/local-ai-lab-agents", "pull", "-q"])   # get the latest fixes
    os.chdir("/content/local-ai-lab-agents")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", *EXTRA], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "projects"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from lal.runtime import setup
MODEL = setup()

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from lal.data import contract, sections, ids, labels
from lal.llm import structured
from lal.checks import quote_is_in_doc, check

## 1. Chunk on structure, with context
Each chunk is one section of one contract. We prepend a short note saying where it came from (Anthropic's "contextual retrieval"), which helps both search methods.

In [ ]:
chunks = []
for cid in ids():
    for s in sections(contract(cid)):
        chunks.append({"id": f"{cid}#{s['id']}", "contract": cid, "section": s["id"],
                       "context": f"From {cid}, section {s['id']} ({s['title']}).", "text": s["text"]})
print(len(chunks), "chunks. Example:", chunks[6])

## 2. Two searches, merged with RRF

In [ ]:
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")      # small, local embedding model
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # what bge models expect on queries
tok = lambda s: re.findall(r"\w+", s.lower())                 # word tokens, so "msa_15," matches "msa_15"

class HybridIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        docs = [f"{c['context']}\n{c['text']}" for c in chunks]
        self.bm25 = BM25Okapi([tok(d) for d in docs])
        self.vecs = embedder.encode(docs, normalize_embeddings=True, show_progress_bar=False)

    def keyword(self, q, n):
        return list(np.argsort(-self.bm25.get_scores(tok(q)))[:n])

    def semantic(self, q, n):
        qv = embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True)[0]
        return list(np.argsort(-(self.vecs @ qv))[:n])

    def search(self, q, k=5, pool=50, mode="hybrid", rrf_k=60):
        if mode == "keyword": return [self.chunks[i] for i in self.keyword(q, k)]
        if mode == "semantic": return [self.chunks[i] for i in self.semantic(q, k)]
        scores = {}
        for ranking in (self.keyword(q, pool), self.semantic(q, pool)):
            for r, i in enumerate(ranking):
                scores[i] = scores.get(i, 0) + 1 / (rrf_k + r + 1)
        return [self.chunks[i] for i in sorted(scores, key=scores.get, reverse=True)[:k]]

index = HybridIndex(chunks)
for c in index.search("msa_15 cap on damages", k=3):
    print(c["id"], "|", c["text"][:80])

## 3. Measure retrieval on its own: recall@k
Did the right chunk come back? We test with questions that use different words from the contract ("cap on damages" vs. "limitation of liability"), and questions with exact IDs.

In [ ]:
QUERIES = []
for cid in ids()[:10]:
    QUERIES += [(f"{cid} cap on damages", f"{cid}#7"), (f"which state's law applies in {cid}", f"{cid}#10"),
                (f"{cid} how do we stop it renewing", f"{cid}#4")]

def recall_at_k(mode, k=3):
    return sum(any(c["id"] == gold for c in index.search(q, k=k, mode=mode)) for q, gold in QUERIES) / len(QUERIES)

for mode in ("keyword", "semantic", "hybrid"):
    print(f"{mode:9} recall@3 = {recall_at_k(mode):.0%}")

## 4. Answer from retrieved chunks, with citations checked in code

In [ ]:
from pydantic import BaseModel
class Answer(BaseModel):
    answer: str
    chunk_id: str
    quote: str

def rag(question, k=4):
    hits = index.search(question, k=k)
    ctx = "\n\n".join(f"[{c['id']}] {c['text']}" for c in hits)
    a = structured([{"role": "system", "content": "Answer only from the sources. Give the source id and an exact quote."},
                    {"role": "user", "content": f"Sources:\n{ctx}\n\nQuestion: {question}"}], Answer)
    cid = a.chunk_id.strip().strip("[]")                     # models often copy the brackets
    src = next((c for c in hits if c["id"] == cid), None)
    a_ok = src is not None and quote_is_in_doc(a.quote, src["text"])
    return a, a_ok

a, ok = rag("What is the liability cap in msa_20?")
print(a.model_dump(), "\ncitation verified:", ok)

In [ ]:
check("hybrid recall@3 is at least as good as keyword-only", recall_at_k("hybrid") >= recall_at_k("keyword"))
check("RAG answer cites a real quote", ok)

## Your turn
1. Remove the context note from each chunk and rerun recall@k. What drops?
2. Add a reranker: `from sentence_transformers import CrossEncoder; CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")`, score the top 20, keep 4.

**Next:** 05 · Tools

## Solutions

In [ ]:
# Solution 1: the same index WITHOUT the context note on each chunk
plain_index = HybridIndex([{**c, "context": ""} for c in chunks])
def recall_with(idx, mode, k=3):
    return sum(any(c["id"] == gold for c in idx.search(q, k=k, mode=mode)) for q, gold in QUERIES) / len(QUERIES)
print(f"{'mode':9} {'with context':>13} {'without':>8}")
for mode in ("keyword", "semantic", "hybrid"):
    print(f"{mode:9} {recall_with(index, mode):>13.0%} {recall_with(plain_index, mode):>8.0%}")
# What to notice: our queries name the contract ("msa_07 cap on damages"), but the chunk text alone never says which
# contract it is from. The context note carries that, which is exactly why contextual retrieval helps.

In [ ]:
# Solution 2: add a reranker: fetch 20, rerank with a cross-encoder, keep the best
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
def search_rerank(q, k=3, first=20):
    cands = index.search(q, k=first)
    scores = reranker.predict([(q, f"{c['context']} {c['text']}") for c in cands])
    return [c for _, c in sorted(zip(scores, cands), key=lambda x: -x[0])][:k]
rr = sum(any(c["id"] == gold for c in search_rerank(q)) for q, gold in QUERIES) / len(QUERIES)
print(f"hybrid recall@3 {recall_at_k('hybrid'):.0%} -> hybrid + rerank recall@3 {rr:.0%}")